# Descriptive statistics

Our analysis is based on the gendered dimension of both journalists and athletes. 
Therefore, we provide descriptive statistics on the distribution of male and female journalists and athletes, as well as a cross-tabulated breakdown.

First, let's unzip transcripts: they contain data on the gender of speakers (journalists).

In [25]:
import zipfile

with zipfile.ZipFile("file.zip", 'r') as zip_ref:
    zip_ref.extractall("./data")

In [39]:
import pandas as pd
import os

# change the path based on the focused sport
data_path = "/home/onyxia/work/PESSD-GBOGC/data/PESSD/curl/"

df = pd.DataFrame()
for file in os.listdir(data_path) : 
    temp = pd.read_csv(data_path+file)
    df = pd.concat([df,temp])

Now, let's join metadata in order to obtain gender of athletes.

In [40]:
df["ID"] = df["audio_file"].str.replace("_wav.wav","")
df = df.merge(pd.read_csv("metadata.csv", sep=","), on="ID")
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath
0,0.00,72.76,"Mais à mon avis c'est de la bonne, c'est de la...",SPEAKER_01,male,CHTP6_wav.wav,CHTP6,H
1,72.76,109.68,Cette Suisse qui nous fait vraiment une impres...,SPEAKER_03,male,CHTP6_wav.wav,CHTP6,H
2,109.68,175.36,"Il faut savoir que Suisse-Suède, c'est un clas...",SPEAKER_01,male,CHTP6_wav.wav,CHTP6,H
3,180.56,188.32,"Alors, c'est les Suisses qui aura remporté l'a...",SPEAKER_03,female,CHTP6_wav.wav,CHTP6,H
4,188.88,260.92,"Alors, le tir à la cible, j'ai regardé Pauline...",SPEAKER_01,male,CHTP6_wav.wav,CHTP6,H
...,...,...,...,...,...,...,...,...
3817,1638.22,1643.22,"Parce que là, elles sont en train de les pouss...",SPEAKER_00,male,CFTP3_wav.wav,CFTP3,F
3818,1643.22,1656.22,"Le but, c'était de faire tout ce trip. Mainten...",SPEAKER_01,male,CFTP3_wav.wav,CFTP3,F
3819,1656.22,1672.22,"Effectivement, c'est un mur de 32 qui... Et ou...",SPEAKER_00,male,CFTP3_wav.wav,CFTP3,F
3820,1672.22,1674.22,"Non, excusez-moi.",SPEAKER_01,male,CFTP3_wav.wav,CFTP3,F


## Descriptive statistics

First, we want to see what is the total duration of the live commentary and how it is distributed among men and women journalists.

In [41]:
# Duration of each segment
df['duration'] = df['stop'] - df['start']

# Total duration
print("Total:")
print(df['duration'].sum() / 60) 

# Journalists
print("Journalists:")
print(df.groupby('main_g')['duration'].sum() / 60) 


Total:
986.6976666666666
Journalists:
main_g
female      307.736000
male        638.789333
music        22.070333
noEnergy      4.607333
noise        13.494667
Name: duration, dtype: float64


Now we want to look at the number of different events based on the gender of athletes (mixt events also are included).

In [29]:
print(df.groupby('g_ath')['duration'].sum() / 60)


g_ath
F    300.594000
H    588.590333
M     97.513333
Name: duration, dtype: float64


Finally, we make a cross tabulation of both journalists' and athletes' gender.

In [30]:

# cross
print("\n Duration (minutes): ")
cross = df.pivot_table(
    values='duration',
    index='main_g',
    columns='g_ath',
    aggfunc='sum'
) / 60
print(cross.round(1))


 Duration (minutes): 
g_ath         F      H     M
main_g                      
female     85.1  221.7   0.9
male      197.8  346.0  95.0
music      12.9    9.1   0.1
noEnergy    1.3    2.7   0.6
noise       3.4    9.1   0.9


Now, we do similar descriptive statistics by replacing time by number of sentences as a unit of analysis.

In [43]:
# dividing texte into sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")


By athletes' gender:

In [32]:
g_ath_counts = df.groupby('g_ath').size()
print(g_ath_counts)
print((g_ath_counts / g_ath_counts.sum() * 100).round(1))

g_ath
F    4077
H    7201
M    1208
dtype: int64
g_ath
F    32.7
H    57.7
M     9.7
dtype: float64


By speakers' gender:

In [33]:
main_g_counts = df.groupby('main_g').size()
print(main_g_counts)
print((main_g_counts / main_g_counts.sum() * 100).round(1))

main_g
female      3844
male        8210
music        224
noEnergy      59
noise        149
dtype: int64
main_g
female      30.8
male        65.8
music        1.8
noEnergy     0.5
noise        1.2
dtype: float64


Cross tabulation:

In [34]:
df_clean = df.reset_index(drop=True) # doublon index issues
print(pd.crosstab(df_clean['main_g'], df_clean['g_ath']))

g_ath        F     H     M
main_g                    
female    1069  2763    12
male      2806  4219  1185
music      135    88     1
noEnergy    18    35     6
noise       49    96     4


## Sentences: descriptive statistics

Now we compare the average length (in words) and their standard deviation for each type of speaker.

In [ ]:
df['text'].str.split().str.len().groupby(df['main_g']).agg(['mean', 'std']).round(1)

,mean,std
main_g,,
female,13.3,11.1
male,11.6,10.5
music,7.6,8.2
noEnergy,5.2,7.4
noise,7.0,7.8


Gender of athletes:

In [45]:
df['text'].str.split().str.len().groupby(df['g_ath']).agg(['mean', 'std']).round(1)

,mean,std
g_ath,,
F,11.6,10.2
H,12.1,11.1
M,12.1,10.0


And in general:

In [44]:
df['text'].str.split().str.len().agg(['mean', 'std']).round(1)

mean    12.0
std     10.7
Name: text, dtype: float64

In [46]:
df['word_count'] = df['text'].str.split().str.len()
print(df['word_count'].describe())
print((df['word_count'] < 5).sum(), "segments de moins de 5 mots")

count    12486.000000
mean        11.960596
std         10.682375
min          0.000000
25%          6.000000
50%          9.000000
75%         15.000000
max        172.000000
Name: word_count, dtype: float64
2181 segments de moins de 5 mots


In [47]:
print((df['word_count'] < 5).sum())
print((df['word_count'] == 0).sum())

2181
10
